# 2.4 Ekspertinė sistema: Medicininė simptomų diagnostika

Šis Colab dokumentas sujungia visus ekspertinės sistemos kodo fragmentus į vieningą veikiančią sistemą.

**Naudojama biblioteka:** `rule-engine` (Python taisyklių mašina)

**Sistemos aprašymas:** Ekspertinė sistema analizuoja paciento simptomus ir pateikia galimas diagnozes naudodama IF-THEN taisykles.

## 1. Bibliotekos įdiegimas

Pirmiausia įdiegiame `rule-engine` biblioteką:

In [ ]:
!pip install rule-engine

In [ ]:
import rule_engine
print("rule-engine versija:", rule_engine.__version__)
print("Biblioteka sėkmingai importuota!")

## 2. Duomenų modelis — Paciento kūrimas

Funkcija `create_patient()` sukuria paciento duomenų žodyną (dict) su visais simptomais.

**Svarbu:** `fever` laukas apskaičiuojamas automatiškai — jei temperatūra > 37.5°C, tai `fever = True`.

In [ ]:
def create_patient(name, age, temperature,
                   cough=False, sore_throat=False, runny_nose=False,
                   headache=False, body_aches=False, fatigue=False,
                   nausea=False, rash=False, shortness_of_breath=False):
    """Sukuria paciento duomenų žodyną su visais simptomais."""
    return {
        "name": name,
        "age": age,
        "temperature": temperature,
        "fever": temperature > 37.5,  # Automatiškai apskaičiuojama
        "cough": cough,
        "sore_throat": sore_throat,
        "runny_nose": runny_nose,
        "headache": headache,
        "body_aches": body_aches,
        "fatigue": fatigue,
        "nausea": nausea,
        "rash": rash,
        "shortness_of_breath": shortness_of_breath,
    }

# Testuojame
test_patient = create_patient("Testas", 30, 38.5, cough=True)
print("Pavyzdinis pacientas:")
for key, value in test_patient.items():
    print(f"  {key}: {value}")

## 3. Taisyklių bazė (Knowledge Base)

Apibrėžiame 10 IF-THEN taisyklių, kiekviena su:
- **Diagnoze** — ligos pavadinimas
- **Tikimybe** — aukšta / vidutinė / žema
- **Taisykle** — `rule-engine` loginė išraiška
- **Rekomendacija** — ką daryti pacientui

In [ ]:
RULES = [
    {
        "diagnosis": "Pneumonija",
        "confidence": "AUKŠTA (SKUBU!)",
        "rule": rule_engine.Rule(
            "fever == true and cough == true and shortness_of_breath == true and body_aches == true"
        ),
        "recommendation": "SKUBIAI kreipkitės į gydytoją arba kvieskite greitąją pagalbą!",
    },
    {
        "diagnosis": "COVID-19 įtarimas",
        "confidence": "VIDUTINĖ",
        "rule": rule_engine.Rule(
            "fever == true and cough == true and fatigue == true and shortness_of_breath == true"
        ),
        "recommendation": "Atlikite COVID-19 testą ir izoliuokitės. Kreipkitės į gydytoją.",
    },
    {
        "diagnosis": "Gripas",
        "confidence": "AUKŠTA",
        "rule": rule_engine.Rule(
            "fever == true and cough == true and body_aches == true and fatigue == true"
        ),
        "recommendation": "Likite namie, gerkite daug skysčių, vartokite antipiretinius vaistus.",
    },
    {
        "diagnosis": "Peršalimas",
        "confidence": "AUKŠTA",
        "rule": rule_engine.Rule(
            "runny_nose == true and sore_throat == true and cough == true and fever == false"
        ),
        "recommendation": "Pailsėkite, gerkite šiltų gėrimų, naudokite nosies lašukus.",
    },
    {
        "diagnosis": "Angina",
        "confidence": "VIDUTINĖ",
        "rule": rule_engine.Rule(
            "sore_throat == true and fever == true and cough == false and runny_nose == false"
        ),
        "recommendation": "Kreipkitės į gydytoją dėl antibiotikų skyrimo.",
    },
    {
        "diagnosis": "Bronchitas",
        "confidence": "VIDUTINĖ",
        "rule": rule_engine.Rule(
            "cough == true and fatigue == true and fever == false"
        ),
        "recommendation": "Venkite šalto oro, gerkite daug skysčių. Jei nepraeina per 2 savaites — kreipkitės į gydytoją.",
    },
    {
        "diagnosis": "Migrena",
        "confidence": "AUKŠTA",
        "rule": rule_engine.Rule(
            "headache == true and nausea == true and fever == false and cough == false"
        ),
        "recommendation": "Pailsėkite tamsioje patalpoje, vartokite skausmalšius.",
    },
    {
        "diagnosis": "Alerginė reakcija",
        "confidence": "VIDUTINĖ",
        "rule": rule_engine.Rule(
            "rash == true and runny_nose == true and fever == false"
        ),
        "recommendation": "Vartokite antihistamininius vaistus. Jei sunku kvėpuoti — skubiai kreipkitės į gydytoją.",
    },
    {
        "diagnosis": "Virškinimo infekcija",
        "confidence": "VIDUTINĖ",
        "rule": rule_engine.Rule(
            "nausea == true and fever == true and cough == false"
        ),
        "recommendation": "Gerkite daug skysčių, laikykitės dietos. Jei nepraeina per 3 dienas — kreipkitės į gydytoją.",
    },
    {
        "diagnosis": "Dehidratacija",
        "confidence": "ŽEMA",
        "rule": rule_engine.Rule(
            "fatigue == true and headache == true and fever == false and cough == false"
        ),
        "recommendation": "Gerkite daugiau vandens ir elektrolitinių gėrimų.",
    },
]

print(f"Sukurta taisyklių: {len(RULES)}")
for i, r in enumerate(RULES, 1):
    print(f"  {i}. {r['diagnosis']} (tikimybė: {r['confidence']})")

## 4. Ekspertinės sistemos variklis (Inference Engine)

Klasė `ExpertSystem` priima paciento duomenis ir tikrina visas taisykles. Grąžina atitinkančias diagnozes.

In [ ]:
class ExpertSystem:
    """Ekspertinė sistema, kuri priima paciento duomenis ir grąžina diagnozes."""

    def __init__(self, rules):
        self.rules = rules

    def diagnose(self, patient):
        """Įvertina visas taisykles ir grąžina atitinkančias diagnozes."""
        diagnoses = []
        for rule_entry in self.rules:
            if rule_entry["rule"].matches(patient):
                diagnoses.append({
                    "diagnosis": rule_entry["diagnosis"],
                    "confidence": rule_entry["confidence"],
                    "recommendation": rule_entry["recommendation"],
                })
        return diagnoses

    def print_report(self, patient):
        """Atspausdina pilną diagnostikos ataskaitą."""
        print(f"\n{'='*60}")
        print(f"PACIENTAS: {patient['name']}")
        print(f"Amžius: {patient['age']} m.")
        print(f"Temperatūra: {patient['temperature']}°C")
        print(f"{'='*60}")

        symptom_names = {
            "fever": "Karščiavimas", "cough": "Kosulys",
            "sore_throat": "Gerklės skausmas", "runny_nose": "Sloga",
            "headache": "Galvos skausmas", "body_aches": "Kūno skausmai",
            "fatigue": "Nuovargis", "nausea": "Pykinimas",
            "rash": "Bėrimas", "shortness_of_breath": "Dusulys",
        }
        symptoms = [label for key, label in symptom_names.items() if patient.get(key, False)]
        print(f"Simptomai: {', '.join(symptoms) if symptoms else 'Nėra'}")
        print("-" * 60)

        diagnoses = self.diagnose(patient)

        if diagnoses:
            print(f"RASTOS DIAGNOZĖS ({len(diagnoses)}):")
            for i, d in enumerate(diagnoses, 1):
                print(f"\n  {i}. {d['diagnosis']}")
                print(f"     Tikimybė: {d['confidence']}")
                print(f"     Rekomendacija: {d['recommendation']}")
        else:
            print("Neįmanoma nustatyti diagnozės pagal pateiktus simptomus.")
            print("Rekomendacija: kreipkitės į gydytoją detalesniems tyrimams.")

        print("=" * 60)
        return diagnoses

# Sukuriame ekspertinės sistemos objektą
expert = ExpertSystem(RULES)
print("Ekspertinė sistema sukurta ir paruošta darbui!")

## 5. Testavimas — 7 pacientai su skirtingais simptomais

Testuojame sistemą su įvairiais simptomų rinkiniais.

### 5.1 Pacientas su gripu

In [ ]:
patient1 = create_patient(
    name="Jonas Jonaitis", age=35, temperature=38.5,
    cough=True, body_aches=True, fatigue=True, headache=True
)
expert.print_report(patient1)

### 5.2 Pacientas su peršalimu

In [ ]:
patient2 = create_patient(
    name="Ona Onaitė", age=28, temperature=36.8,
    runny_nose=True, sore_throat=True, cough=True
)
expert.print_report(patient2)

### 5.3 Pacientas su pneumonija (SKUBU!)

In [ ]:
patient3 = create_patient(
    name="Petras Petraitis", age=65, temperature=39.2,
    cough=True, body_aches=True, fatigue=True, shortness_of_breath=True
)
expert.print_report(patient3)

### 5.4 Pacientas su migrena

In [ ]:
patient4 = create_patient(
    name="Birutė Birutienė", age=42, temperature=36.6,
    headache=True, nausea=True
)
expert.print_report(patient4)

### 5.5 Pacientas su alergine reakcija

In [ ]:
patient5 = create_patient(
    name="Antanas Antanaitis", age=22, temperature=36.9,
    rash=True, runny_nose=True
)
expert.print_report(patient5)

### 5.6 Pacientas su COVID-19 įtarimu

In [ ]:
patient6 = create_patient(
    name="Ieva Ievaitė", age=50, temperature=38.8,
    cough=True, fatigue=True, shortness_of_breath=True
)
expert.print_report(patient6)

### 5.7 Pacientas be aiškios diagnozės

In [ ]:
patient7 = create_patient(
    name="Dainius Dainauskas", age=30, temperature=36.5,
    headache=True
)
expert.print_report(patient7)

## 6. Santrauka

| Pacientas | Simptomai | Diagnozė |
|-----------|-----------|----------|
| Jonas | Karščiavimas, kosulys, kūno skausmai, nuovargis | Gripas |
| Ona | Sloga, gerklės skausmas, kosulys | Peršalimas |
| Petras | Karščiavimas, kosulys, kūno skausmai, dusulys | Pneumonija + COVID-19 + Gripas |
| Birutė | Galvos skausmas, pykinimas | Migrena |
| Antanas | Bėrimas, sloga | Alerginė reakcija |
| Ieva | Karščiavimas, kosulys, nuovargis, dusulys | COVID-19 įtarimas |
| Dainius | Galvos skausmas | Nenustatyta |

---
**PASTABA:** Ši sistema yra skirta mokymosi tikslams. Ji NEPAKEIČIA profesionalios medicinos konsultacijos!